# Exercise 3: Neural networks in PyTorch

In this exercise you’ll implement small neural-network building blocks from scratch and use them to train a simple classifier.

You’ll cover:
- **Basic layers**: Linear, Embedding, Dropout
- **Normalization**: LayerNorm and RMSNorm
- **MLPs + residual**: composing layers into deeper networks
- **Classification**: generating a learnable dataset, implementing cross-entropy from logits, and writing a minimal training loop

As before: fill in all `TODO`s without changing function names or signatures.
Use small sanity checks and compare to PyTorch reference implementations when useful.

In [1]:
from __future__ import annotations

import torch
from torch import nn

## Basic layers

In this section you’ll implement a few core layers that appear everywhere:

### `Linear`
A fully-connected layer that follows nn.Linear conventions:  
`y = x @ Wᵀ + b`

Important details:
- Parameters should be registered as `nn.Parameter`
- Store weight as (out_features, in_features) like nn.Linear.
- The forward pass should support leading batch dimensions: `x` can be shape `(..., in_features)`

### `Embedding`
An embedding table maps integer ids to vectors:
- input: token ids `idx` of shape `(...,)`
- output: vectors of shape `(..., embedding_dim)`

This is essentially a learnable lookup table.

### `Dropout`
Dropout randomly zeroes activations during training to reduce overfitting.
Implementation details:
- Only active in `model.train()` mode
- In training: drop with probability `p` and scale the kept values by `1/(1-p)` so the expected value stays the same
- In eval: return the input unchanged

## Instructions
- Do not use PyTorch reference modules for the parts you implement (e.g. don’t call nn.Linear inside your Linear).
- You may use standard tensor ops that you learned before (matmul, sum, mean, rsqrt, indexing, etc.).
- Use a parameter initialization method of your choice. We recommend something like Xavier-uniform.


In [2]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias_param = nn.Parameter(torch.empty(out_features)) if bias else None
        # Xavier uniform init
        bound = (6.0 / (in_features + out_features)) ** 0.5
        nn.init.uniform_(self.weight, -bound, bound)
        if self.bias_param is not None:
            nn.init.zeros_(self.bias_param)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (..., in_features)
        return: (..., out_features)
        """
        out = x @ self.weight.t()
        if self.bias_param is not None:
            out = out + self.bias_param
        return out

lin = Linear(4, 8)
x = torch.randn(3, 4)
print(lin(x).shape)

torch.Size([3, 8])


In [3]:
class Embedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_embeddings, embedding_dim) * 0.02)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """
        idx: (...,) int64
        return: (..., embedding_dim)
        """
        return self.weight[idx]

emb = Embedding(10, 16)
idx = torch.tensor([0, 3, 5])
print(emb(idx).shape)

torch.Size([3, 16])


In [4]:
class Dropout(nn.Module):
    def __init__(self, p: float):
        super().__init__()
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        In train mode: drop with prob p and scale by 1/(1-p).
        In eval mode: return x unchanged.
        """
        if not self.training or self.p == 0.0:
            return x
        mask = torch.bernoulli(torch.full_like(x, 1 - self.p))
        return x * mask / (1 - self.p)

drop = Dropout(0.5)
drop.train()
x = torch.ones(4, 8)
print(drop(x).mean().item())
drop.eval()
print(drop(x).mean().item())

1.0625
1.0


## Normalization

Normalization layers help stabilize training by controlling activation statistics.

### LayerNorm
LayerNorm normalizes each example across its **feature dimension** (the last dimension):

- compute mean and variance over the last dimension
- normalize: `(x - mean) / sqrt(var + eps)`
- apply learnable per-feature scale and shift (`weight`, `bias`)

**In this exercise, assume `elementwise_affine=True` (always include `weight` and `bias`).**  
`weight` and `bias` each have shape `(D,)`.

LayerNorm is widely used in transformers because it does not depend on batch statistics.

### RMSNorm
RMSNorm is similar to LayerNorm but normalizes using only the root-mean-square:
- `x / sqrt(mean(x^2) + eps)` over the last dimension
- usually includes a learnable scale (`weight`)
- no mean subtraction

RMSNorm is popular in modern LLMs because it's faster.


In [5]:
class LayerNorm(nn.Module):
    def __init__(
        self, normalized_shape: int, eps: float = 1e-5, elementwise_affine: bool = True
    ):
        super().__init__()
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if elementwise_affine:
            self.weight = nn.Parameter(torch.ones(normalized_shape))
            self.bias = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Normalize over the last dimension.
        x: (..., D)
        """
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / (var + self.eps).sqrt()
        if self.elementwise_affine:
            x_norm = x_norm * self.weight + self.bias
        return x_norm

ln = LayerNorm(8)
x = torch.randn(3, 8)
out = ln(x)
print(out.shape, out.mean(dim=-1).abs().max().item())

# compare to torch reference
ref = nn.LayerNorm(8)
ref.weight.data = ln.weight.data.clone()
ref.bias.data = ln.bias.data.clone()
print(torch.allclose(out, ref(x), atol=1e-5))

torch.Size([3, 8]) 4.470348358154297e-08
True


In [6]:
class RMSNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        RMSNorm: x / sqrt(mean(x^2) + eps) * weight
        over the last dimension.
        """
        rms = (x.pow(2).mean(dim=-1, keepdim=True) + self.eps).sqrt()
        return (x / rms) * self.weight

rms = RMSNorm(8)
x = torch.randn(3, 8)
out = rms(x)
print(out.shape)

torch.Size([3, 8])


## MLPs and residual networks

Now you’ll build larger networks by composing layers.

### MLP
An MLP is a stack of `depth` Linear layers with non-linear activations (use GELU) in between.
In this exercise you’ll support:
- configurable depth
- a hidden dimension
- optional LayerNorm between layers (a common stabilization trick)

A key skill is building networks using `nn.ModuleList` / `nn.Sequential` while keeping shapes consistent.

### Transformer-style FeedForward (FFN)
A transformer block contains a position-wise feedforward network:
- `D -> 4D -> D` (by default)
- activation is typically **GELU**

This is essentially an MLP applied independently at each token position.

### Residual wrapper
Residual connections are the simplest form of “skip connection”:
- output is `x + fn(x)`

They improve gradient flow and allow training deeper networks more reliably.

In [7]:
class MLP(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        out_dim: int,
        depth: int,
        use_layernorm: bool = False,
    ):
        super().__init__()
        layers = []
        current_dim = in_dim
        for i in range(depth):
            next_dim = out_dim if i == depth - 1 else hidden_dim
            layers.append(nn.Linear(current_dim, next_dim))
            if i < depth - 1:
                if use_layernorm:
                    layers.append(nn.LayerNorm(next_dim))
                layers.append(nn.GELU())
            current_dim = next_dim
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

mlp = MLP(4, 16, 2, depth=3, use_layernorm=True)
x = torch.randn(8, 4)
print(mlp(x).shape)

torch.Size([8, 2])


In [8]:
class FeedForward(nn.Module):
    """
    Transformer-style FFN: D -> 4D -> D (default)
    """

    def __init__(self, d_model: int, d_ff: int | None = None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.act(self.fc1(x)))

ffn = FeedForward(16)
x = torch.randn(3, 5, 16)
print(ffn(x).shape)

torch.Size([3, 5, 16])


In [ ]:
class Residual(nn.Module):
    def __init__(self, fn: nn.Module):
        super().__init__()
        self.fn = fn

    def forward(self, x: torch.Tensor, *args, **kwargs) -> torch.Tensor:
        return x + self.fn(x, *args, **kwargs)

res = Residual(FeedForward(16))
x = torch.randn(3, 5, 16)
print(res(x).shape)

## Classification problem

In this section you’ll put everything together in a minimal MNIST classification experiment.

You will:
1) download and load the MNIST dataset
2) implement cross-entropy from logits (stable, using log-softmax)
3) build a simple MLP-based classifier (flatten MNIST images first)
4) write a minimal training loop
5) report train loss curve and final accuracy

The goal here is not to reach state-of-the-art accuracy, but to understand the full pipeline:
data → model → logits → loss → gradients → parameter update.

### Model notes
- We want you to combine the MLP we implemented above with the classification head we define below into one model 

### MNIST notes
- MNIST images are `28×28` grayscale.
- After `ToTensor()`, each image has shape `(1, 28, 28)` and values in `[0, 1]`.
- For an MLP classifier, we flatten to a vector of length `784`.

## Deliverables
- Include a plot of your train loss curve in the video submission as well as a final accuracy. 
- **NOTE** Here we don't grade on model performance but we expect you to achieve at least 70% accuracy to confirm a correct model implementation.

In [10]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [11]:
transform = transforms.ToTensor()  # -> float32 in [0,1], shape (1, 28, 28)

train_ds = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)

print(f"Train size: {len(train_ds)}, Test size: {len(test_ds)}")


0.3%


0.7%


1.0%


1.3%


1.7%

...


97.4%


99.4%


100.0%


100.0%

Train size: 60000, Test size: 10000


In [12]:
def cross_entropy_from_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
) -> torch.Tensor:
    """
    Compute mean cross-entropy loss from logits.

    logits: (B, C)
    targets: (B,) int64

    Requirements:
    - Use log-softmax for stability (do not use torch.nn.CrossEntropyLoss, we check this in the autograder).
    """
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
    loss = -log_probs[torch.arange(len(targets)), targets]
    return loss.mean()

logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))
print(cross_entropy_from_logits(logits, targets))

tensor(2.6173)


In [13]:
class ClassificationHead(nn.Module):
    def __init__(self, d_in: int, num_classes: int):
        super().__init__()
        self.linear = nn.Linear(d_in, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (..., d_in)
        return: (..., num_classes) logits
        """
        return self.linear(x)

head = ClassificationHead(16, 10)
print(head(torch.randn(3, 16)).shape)

torch.Size([3, 10])


In [14]:
def accuracy(loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.view(x.shape[0], -1))
            preds = logits.argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.shape[0]
    return correct / total

In [15]:
def train_classifier(
    model: nn.Module,
    train_data_loader: DataLoader,
    test_data_loader: DataLoader,
    lr: float,
    epochs: int,
    seed: int = 0,
) -> list[float]:
    """
    Minimal training loop for MNIST classification.

    Steps:
    - define optimizer
    - for each epoch:
        - sample minibatches
        - forward -> cross-entropy -> backward -> optimizer step
      - compute test accuracy at the end of each epoch
    - return list of training losses (one per update step)

    Requirements:
    - call model.train() during training and model.eval() during evaluation
    - do not use torch.nn.CrossEntropyLoss (use your cross_entropy_from_logits)
    """
    torch.manual_seed(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []

    for epoch in range(epochs):
        model.train()
        for x, y in train_data_loader:
            x_flat = x.view(x.shape[0], -1)
            logits = model(x_flat)
            loss = cross_entropy_from_logits(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        model.eval()
        acc = accuracy(test_data_loader)
        print(f"Epoch {epoch+1}/{epochs} | Test Accuracy: {acc:.4f}")

    return losses


# Build and train the model
model = nn.Sequential(
    MLP(784, 256, 256, depth=3, use_layernorm=True),
    ClassificationHead(256, 10),
)

losses = train_classifier(model, train_loader, test_loader, lr=1e-3, epochs=5, seed=0)
print(f"Final test accuracy: {accuracy(test_loader):.4f}")

Epoch 1/5 | Test Accuracy: 0.9686


Epoch 2/5 | Test Accuracy: 0.9739


Epoch 3/5 | Test Accuracy: 0.9720


Epoch 4/5 | Test Accuracy: 0.9757


Epoch 5/5 | Test Accuracy: 0.9775


Final test accuracy: 0.9775


In [16]:
print(f"Training losses (first 5): {losses[:5]}")
print(f"Training losses (last 5): {losses[-5:]}")
print(f"Final test accuracy: {accuracy(test_loader):.4f}")

Training losses (first 5): [2.299438953399658, 1.938456654548645, 1.692189335823059, 1.4938427209854126, 1.2313573360443115]
Training losses (last 5): [0.008279120549559593, 0.055932994931936264, 0.04866769164800644, 0.02446119859814644, 0.06952717155218124]


Final test accuracy: 0.9775
